In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv('data/cleaned_data.csv')
data.head()

,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,fat_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,Banana Chips Sweetened (Whole),not mentioned,Not Mentioned,Labels are missing,"Bananas, vegetable oil (coconut oil, corn oil ...",unknown,No additives,d,2243.0,28.57,...,0.0,no information,14.0,14.0,Not Mentioned,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified
1,Peanuts,torn & glasser,Not Mentioned,Labels are missing,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:soy, en:wheat, en:peanuts",No additives,b,1941.0,17.86,...,0.0,no information,0.0,0.0,Not Mentioned,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified
2,Organic Salted Nut Mix,grizzlies,Not Mentioned,Labels are missing,"Organic hazelnuts, organic cashews, organic wa...",unknown,No additives,d,2540.0,57.14,...,0.0,no information,12.0,12.0,Not Mentioned,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified
3,Organic Polenta,bob's red mill,Not Mentioned,Labels are missing,Organic polenta,unknown,No additives,not given,1552.0,1.43,...,0.0,no information,not given,not given,Not Mentioned,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified
4,Breadshop Honey Gone Nuts Granola,unfi,Not Mentioned,Labels are missing,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,18.27,...,0.0,no information,not given,not given,Not Mentioned,Not Specified,Not Specified,Not Specified,Not Specified,Not Specified


In [3]:
us_data = pd.read_csv('data/us_data.csv')
us_data.shape

(171521, 98)

In [4]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].apply(clean_ingredients)
us_data['ingredients'] = us_data['ingredients_text'].apply(clean_ingredients)

In [5]:
data = data.drop(columns=['ingredients_text', 'categories_en'])
us_data = us_data.drop(columns=['ingredients_text', 'categories_en'])

### Pre-processing 

In [6]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['product_name'] = data['product_name'].apply(clean_text)
data['ingredients'] = data['ingredients'].apply(clean_text)
data['allergens_en'] = data['allergens_en'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

In [7]:
# Replace placeholders with NaN or empty lists
data['allergens_en'] = data['allergens_en'].replace('unknown', np.nan)
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)

In [8]:
# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')
data['allergens_en'] = data['allergens_en'].str.split(', ')

### Implement product matching

In [18]:
from fuzzywuzzy import process

# Function to find top 5 closest matches
def find_top_matches(user_input, choices, limit=5):
    matches = process.extract(user_input, choices, limit=limit)
    return matches

# Example: User inputs a product name
user_input = "Hamburger buns"
top_matches = find_top_matches(user_input, data['product_name'].tolist())

print(f"Top matches for '{user_input}':")
for match, score in top_matches:
    print(f"- {match} (Score: {score})")

Top matches for 'Hamburger buns':
- hamburger buns (Score: 100)
- hamburger buns (Score: 100)
- hamburger buns (Score: 100)
- hamburger buns (Score: 100)
- hamburger buns (Score: 100)


### Category filtering

In [19]:
# Function to find the first known category from top matches
def find_known_category(top_matches, df):
    for match, score in top_matches:
        match_categories = df[df['product_name'] == match][['category_level_1', 'category_level_2', 'category_level_3', 'category_level_4', 'category_level_5']].values[0]
        if match_categories[0] != 'not mentioned':
            return match_categories
    return None

# Find the first known category
selected_category = find_known_category(top_matches, data)

if selected_category is not None:
    print(f"Using category from match: {selected_category}")
else:
    print("No known category found in top matches.")

Using category from match: ['plant-based foods and beverages' 'plant-based foods'
 'Cereals and potatoes' 'Breads' 'Special breads']


In [20]:
# Function to get the most common category
def get_default_category(df):
    return df[df.category_level_1 != 'not mentioned']['category_level_1'].mode()[0]

# Use the selected category or fallback to default
if selected_category is None:
    default_category = get_default_category(data)
    selected_category = [default_category, 'Not Specified']
    print(f"Warning: No known category found in top matches. Using default category: {default_category}.")

### Allergens filtering & Recommendations

In [22]:
# Function to filter products by allergens
def filter_by_allergens(products, allergens_to_avoid):
    for allergen in allergens_to_avoid:
        if f'contains_{allergen}' in products.columns:
            products = products[~products[f'contains_{allergen}']]
    return products

# Filter products in the selected category
same_category_products = data[
    (data['category_level_1'] == selected_category[0]) & 
    (data['category_level_2'] == selected_category[1]) &
    (data['category_level_3'] == selected_category[2]) &
    (data['category_level_4'] == selected_category[3]) &
    (data['category_level_5'] == selected_category[4])
]

# Filter by allergens
allergens_to_avoid = []  # Example allergens
filtered_products = filter_by_allergens(same_category_products, allergens_to_avoid)

# Limit to top N recommendations
top_n = 5
recommendations = filtered_products[['product_name', 'additives_en']].head(top_n)

print("Final recommendations:")
print(recommendations)

Final recommendations:
                       product_name  \
1593                 hamburger buns   
1648  jumbo hamburger enriched buns   
1815        original hamburger buns   
1921  white hamburger enriched buns   
6277           wheat hamburger buns   

                                           additives_en  
1593                          E101,E101i,E282,E375,E471  
1648                                    E101,E101i,E375  
1815                               E101,E101i,E375,E481  
1921  E101,E101i,E282,E300,E375,E412,E471,E481,E516,...  
6277  E101,E101i,E150a,E282,E322,E322i,E341,E341i,E3...  
